# Math-Verify 解析与判分 Walkthrough

这个 notebook 直接调用仓库默认的 `scripts/parser.py::parse_v5_dual_and_verify()`，用于直观看到：

1. 当前能够正确处理的常见答案形式；
2. 典型错误、解析失败与系统异常如何分类；
3. 当前刻意不放宽或无法可靠处理的边界。

它不会调用模型或占用 GPU，也没有在 notebook 中重新实现解析逻辑。每组案例都有预期状态断言；行为变化时，重新运行会直接失败。

In [ ]:
from collections import Counter
from dataclasses import asdict
from html import escape
from pathlib import Path
import sys
from unittest.mock import patch

from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / "scripts" / "parser.py").is_file():
    raise RuntimeError("请从 math-eval 仓库根目录运行这个 notebook")
sys.path.insert(0, str(ROOT / "scripts"))

import parser as parser_module

print(f"parser_id: {parser_module.V5_DUAL_PARSER_ID}")
print(f"math-verify: {parser_module.MATH_VERIFY_VERSION}")
print(f"parser_config_hash: {parser_module.V5_DUAL_PARSER_CONFIG_HASH}")

## 如何阅读结果

- `strict`：正式分数，只接受最后一个完整的 `\boxed{}`。
- `soft`：没有完整 strict box 时，把非空全文交给 Math-Verify，仅作诊断。
- `candidate_text`：对应模式实际交给 Math-Verify 的候选文本。
- `normalized_prediction` / `normalized_gold`：Math-Verify 解析后的表达式候选。
- `last_boxed`：strict 选择的最后一个完整 `\boxed{}`；`\fbox{}` 不属于 strict 协议。
- `full_final_text`：没有完整 box 时，把全文交给上游提取。
- `incorrect`：解析成功，但数学比较不等价。
- `no_candidate`、`parse_error`、`verification_error`：三类 parser failure；统计正确率时仍全部算错。

In [ ]:
TABLE_FIELDS = (
    "name",
    "gold",
    "final_text",
    "strict_status",
    "soft_status",
    "strict_candidate",
    "soft_candidate",
    "note",
)

def _cell(value):
    rendered = "—" if value is None else escape(str(value))
    return f"<td><code style='white-space:pre-wrap'>{rendered}</code></td>"

def show_cases(cases):
    rows = []
    for case in cases:
        result = asdict(parser_module.parse_v5_dual_and_verify(case["final_text"], case["gold"]))
        row = {
            **case,
            "strict_status": result["strict"]["status"],
            "soft_status": result["soft"]["status"],
            "strict_candidate": result["strict"]["candidate_text"],
            "soft_candidate": result["soft"]["candidate_text"],
        }
        rows.append(row)
        assert row["strict_status"] == case["expected_strict"], (case["name"], row)
        assert row["soft_status"] == case["expected_soft"], (case["name"], row)

    header = "".join(f"<th>{escape(field)}</th>" for field in TABLE_FIELDS)
    body = "".join(
        "<tr>" + "".join(_cell(row.get(field)) for field in TABLE_FIELDS) + "</tr>"
        for row in rows
    )
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"))
    return rows

## 1. 当前正确处理的答案形式

这些案例覆盖 boxed/fbox、分数、小数、代数等价、tuple、复数、自然语言全文提取，以及末尾不完整 box。fbox 和无 box 全文只可能被 soft 恢复。

In [ ]:
correct_cases = [
    {"name": "boxed integer", "gold": "42", "final_text": r"\boxed{42}", "expected_strict": "correct", "expected_soft": "correct", "note": "完整 boxed"},
    {"name": "fbox fraction", "gold": "0.5", "final_text": r"\fbox{\frac{1}{2}}", "expected_strict": "no_candidate", "expected_soft": "correct", "note": "fbox 仅被 soft 恢复"},
    {"name": "algebra", "gold": "(x-1)(x+1)", "final_text": r"\boxed{x^2-1}", "expected_strict": "correct", "expected_soft": "correct", "note": "符号化简后等价"},
    {"name": "tuple", "gold": r"\left(3, \frac{\pi}{2}\right)", "final_text": r"\boxed{(3, \frac{\pi}{2})}", "expected_strict": "correct", "expected_soft": "correct", "note": "tuple 元素逐项比较"},
    {"name": "complex", "gold": "6 - 5i", "final_text": r"\boxed{6-5i}", "expected_strict": "correct", "expected_soft": "correct", "note": "复数表达式"},
    {"name": "upstream tolerance", "gold": r"\frac{1}{3}", "final_text": r"\boxed{0.333333}", "expected_strict": "correct", "expected_soft": "correct", "note": "保持 Math-Verify 0.9.0 默认容差"},
    {"name": "full text", "gold": "12", "final_text": "The final answer is $12$.", "expected_strict": "no_candidate", "expected_soft": "correct", "note": "无 box，仅 soft 使用全文"},
    {"name": "multiple boxes", "gold": "2", "final_text": r"\boxed{1}\boxed{2}", "expected_strict": "correct", "expected_soft": "correct", "note": "选择最后一个完整 box"},
    {"name": "trailing incomplete box", "gold": "2", "final_text": r"\boxed{1}\boxed{2}\boxed{\frac{3}{", "expected_strict": "correct", "expected_soft": "correct", "note": "忽略末尾未闭合 box"},
]
correct_results = show_cases(correct_cases)

## 2. 典型错误与失败分类

相邻小数反例用于证明本仓库没有自行放宽上游容差。无完整 box 时 strict 为 `no_candidate`；soft 再区分空文本和全文解析失败。

In [ ]:
error_cases = [
    {"name": "outside tolerance", "gold": r"\frac{1}{3}", "final_text": r"\boxed{0.3333}", "expected_strict": "incorrect", "expected_soft": "incorrect", "note": "上游默认判为不等价"},
    {"name": "wrong nearby value", "gold": r"\frac{1}{3}", "final_text": r"\boxed{0.34}", "expected_strict": "incorrect", "expected_soft": "incorrect", "note": "明确错误近邻"},
    {"name": "wrong integer", "gold": "42", "final_text": r"\boxed{41}", "expected_strict": "incorrect", "expected_soft": "incorrect", "note": "解析成功但不等价"},
    {"name": "wrong algebra", "gold": "(x-1)(x+1)", "final_text": r"\boxed{x^2+1}", "expected_strict": "incorrect", "expected_soft": "incorrect", "note": "符号化简后仍不等价"},
    {"name": "empty output", "gold": "42", "final_text": "", "expected_strict": "no_candidate", "expected_soft": "no_candidate", "note": "没有候选文本"},
    {"name": "no answer", "gold": "42", "final_text": "No final answer was produced.", "expected_strict": "no_candidate", "expected_soft": "parse_error", "note": "soft 全文无法提取数学候选"},
]
error_results = show_cases(error_cases)

### `verification_error` 的分类演示

`verification_error` 不是一种数学答案，而是上游验证器异常。下面只用 Python 标准库临时模拟异常，确认当前 parser 会把它单独记录，而不是混成 `incorrect`。

In [ ]:
with patch.object(parser_module, "verify", side_effect=RuntimeError("demo failure")):
    verification_result = asdict(parser_module.parse_v5_dual_and_verify(r"\boxed{42}", "42"))
verification_failure = {
    "name": "verification error",
    "strict_status": verification_result["strict"]["status"],
    "soft_status": verification_result["soft"]["status"],
}
assert verification_failure["strict_status"] == "verification_error"
assert verification_failure["soft_status"] == "verification_error"
display(verification_result)

## 3. 当前已知边界

以下表达式在人类结合题目语境时可能被视为同一个答案，但 Math-Verify 0.9.0 当前按结构严格拒绝。本仓库刻意不增加跨类型特判。

In [ ]:
boundary_cases = [
    {"name": "matrix vs tuple", "gold": "(1,2)", "final_text": r"\boxed{\begin{pmatrix}1\\2\end{pmatrix}}", "expected_strict": "incorrect", "expected_soft": "incorrect", "note": "不自动假设列向量等于 tuple"},
    {"name": "set-builder vs interval", "gold": r"(0,\infty)", "final_text": r"\boxed{\{x \mid x > 0\}}", "expected_strict": "incorrect", "expected_soft": "incorrect", "note": "v5 正确匹配转义花括号，但不放宽集合等价"},
]
boundary_results = show_cases(boundary_cases)

另外还有三类不在当前自动判分承诺内：

- 选择题字母：当前没有启用 `StringExtractionConfig`。
- 依赖题目语境才能确认的等价，例如单位、方向、隐含定义。
- gold 本身错误、含糊或无法解析；这属于 canonical 数据错误，不应记成模型错误。

这些边界应先通过真实数据复现并反馈上游，而不是在本仓库累积题目特判。

## 汇总

这里分别展示 strict/soft 状态计数与 failure 比例。正式评测使用 strict；两种模式的所有非 `correct` 状态都进入各自分母。

In [ ]:
all_results = correct_results + error_results + [verification_failure] + boundary_results
summary = {"sample_count": len(all_results)}
for mode in ("strict", "soft"):
    counts = Counter(row[f"{mode}_status"] for row in all_results)
    failures = sum(counts[name] for name in ("no_candidate", "parse_error", "verification_error"))
    summary[mode] = {
        "status_counts": dict(sorted(counts.items())),
        "accuracy": counts["correct"] / len(all_results),
        "evaluation_failure_rate": failures / len(all_results),
    }
display(summary)
assert len(all_results) == 18
assert summary["strict"]["accuracy"] <= summary["soft"]["accuracy"]
print("全部 walkthrough 断言通过。")